In [1]:
import os
import glob
from docx import Document
from tqdm import tqdm  # Use tqdm.notebook for Jupyter Notebooks
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

def read_docx(file_path):
    doc = Document(file_path)
    lines = []
    for para in doc.paragraphs:
        para_lines = para.text.split('\n')
        lines.extend(para_lines)
    return lines

def extract_subject_from_filename(filename):
    if 'Maths' in filename:
        return 'Maths Statement or Equation'
    elif 'Chemistry' in filename:
        return 'Chemistry Statement'
    elif 'General' in filename:
        return 'General Statement'

def generate_questions_batch(scenarios, generate_text, subject, batch_size=5):
    questions_batch = []
    total_scenarios = len(scenarios)
    for i in tqdm(range(0, total_scenarios, batch_size), desc="Generating Questions", leave=False):
        batch = scenarios[i:i+batch_size]
        prompts = [
            f"""[INST]Imagine you are a human, this is the first time you are coming across this {subject}, you have no previous knowledge of it "{scenario}", what are the top 5 questions that would pop up in your head which would be most useful in learning about it as you are new to it. Give me a simple bullet point list, don't explain them or expand them[/INST]"""
            for scenario in batch
        ]
        results = generate_text(prompts)
        for result in results:
            questions_batch.append(result[0]["generated_text"])

    return questions_batch

def write_questions_to_file(questions, output_file_path):
    with open(output_file_path, "a", encoding="utf-8") as file:
        for question in questions:
            file.write(question + "\n\n")

def process_directory(input_directory, output_directory, batch_size=5):
    os.makedirs(output_directory, exist_ok=True)
    
    tokenizer = AutoTokenizer.from_pretrained("models--meta-llama--Llama-2-7b-chat-hf/snapshots/c1b0db933684edbfe29a06fa47eb19cc48025e93", local_files_only=True)
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained("models--meta-llama--Llama-2-7b-chat-hf/snapshots/c1b0db933684edbfe29a06fa47eb19cc48025e93", local_files_only=True, quantization_config=bnb_config)
    generate_text = pipeline(model=model, tokenizer=tokenizer, return_full_text=True, task='text-generation', temperature=0.1, max_new_tokens=128, repetition_penalty=1.1)
    
    docx_files = glob.glob(os.path.join(input_directory, '*.docx'))
    total_files = len(docx_files)
    for file_path in tqdm(docx_files, desc="Processing Files"):
        subject = extract_subject_from_filename(os.path.basename(file_path))
        lines = read_docx(file_path)
        tqdm.write(f"Processing {os.path.basename(file_path)} with {len(lines)} sentences. Subject: {subject}")
        questions = generate_questions_batch(lines, generate_text, subject, batch_size=batch_size)
        
        output_file_name = os.path.basename(file_path).replace('.docx', '_results.txt')
        output_file_path = os.path.join(output_directory, output_file_name)
        
        write_questions_to_file(questions, output_file_path)
        tqdm.write(f"Finished processing {file_path}. Results appended to {output_file_path}\n")

# Example usage
process_directory('Question_data_2', 'outputs_7b', batch_size=5)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Files:   0%|          | 0/7 [00:00<?, ?it/s]

Processing Chemistry - Intermediate.docx with 161 sentences. Subject: Chemistry Statement



Generating Questions:  30%|███       | 10/33 [04:05<09:28, 24.71s/it]/home/sjavaji_umass_edu/.local/lib/python3.9/site-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(

Processing Files:  14%|█▍        | 1/7 [13:34<1:21:24, 814.15s/it]   

Finished processing Question_data_2/Chemistry - Intermediate.docx. Results appended to outputs_7b/Chemistry - Intermediate_results.txt

Processing Chemistry BASIC.docx with 161 sentences. Subject: Chemistry Statement


...

Finished processing Question_data_2/Maths - BASIC statements.docx. Results appended to outputs_7b/Maths - BASIC statements_results.txt

Processing Maths - Intermediate .docx with 108 sentences. Subject: Maths Statement or Equation



Processing Files:  86%|████████▌ | 6/7 [1:22:43<12:39, 759.77s/it]   

Finished processing Question_data_2/Maths - Intermediate .docx. Results appended to outputs_7b/Maths - Intermediate _results.txt

Processing Maths -Advance.docx with 101 sentences. Subject: Maths Statement or Equation



Processing Files: 100%|██████████| 7/7 [1:31:04<00:00, 780.60s/it]   

Finished processing Question_data_2/Maths -Advance.docx. Results appended to outputs_7b/Maths -Advance_results.txt



In [1]:
import os
import glob
from docx import Document
from tqdm import tqdm  # Use tqdm.notebook for Jupyter Notebooks
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

def read_docx(file_path):
    doc = Document(file_path)
    lines = []
    for para in doc.paragraphs:
        para_lines = para.text.split('\n')
        lines.extend(para_lines)
    return lines

def generate_questions_batch(scenarios, generate_text, batch_size=5):
    questions_batch = []
    total_scenarios = len(scenarios)
    for i in tqdm(range(0, total_scenarios, batch_size), desc="Generating Questions", leave=False):
        batch = scenarios[i:i+batch_size]
        prompts = [
            f"""[INST]Imagine you are a human, this is the first time you are coming across this {subject} statement, you have no previous knowledge of it "{scenario}", what are the top 5 questions that would pop up in your head which would be most useful in learning about it as you are new to it. Give me a simple bullet point list, don't explain them or expand them[/INST]"""
            for scenario in batch
        ]
        results = generate_text(prompts)
        for result in results:
            questions_batch.append(result[0]["generated_text"])

    return questions_batch

def write_questions_to_file(questions, output_file_path):
    with open(output_file_path, "a", encoding="utf-8") as file:
        for question in questions:
            file.write(question + "\n\n")

def process_directory(input_directory, output_directory, batch_size=5):
    os.makedirs(output_directory, exist_ok=True)
    
    tokenizer = AutoTokenizer.from_pretrained("models--meta-llama--Llama-2-7b-chat-hf/snapshots/c1b0db933684edbfe29a06fa47eb19cc48025e93", local_files_only=True)
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained("models--meta-llama--Llama-2-7b-chat-hf/snapshots/c1b0db933684edbfe29a06fa47eb19cc48025e93", local_files_only=True, quantization_config=bnb_config)
    generate_text = pipeline(model=model, tokenizer=tokenizer, return_full_text=True, task='text-generation', temperature=0.1, max_new_tokens=128, repetition_penalty=1.1)
    
    docx_files = glob.glob(os.path.join(input_directory, '*.docx'))
    total_files = len(docx_files)
    for file_path in tqdm(docx_files, desc="Processing Files"):
        lines = read_docx(file_path)
        tqdm.write(f"Processing {os.path.basename(file_path)} with {len(lines)} sentences.")
        questions = generate_questions_batch(lines, generate_text, batch_size=batch_size)
        
        output_file_name = os.path.basename(file_path).replace('.docx', '_results.txt')
        output_file_path = os.path.join(output_directory, output_file_name)
        
        write_questions_to_file(questions, output_file_path)
        tqdm.write(f"Finished processing {file_path}. Results appended to {output_file_path}\n")

# Update the paths to the tokenizer and model as necessary.
# Example usage
process_directory('Question_data_2', 'outputs_7b', batch_size=5)


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Processing Files:   0%|          | 0/6 [00:00<?, ?it/s]

Processing MATHS -Advance.docx with 101 sentences.



Generating Questions:  48%|████▊     | 10/21 [11:58<13:24, 73.12s/it]/home/sjavaji_umass_edu/.local/lib/python3.9/site-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(

Processing Files:  17%|█▋        | 1/6 [24:01<2:00:07, 1441.55s/it]  

Finished processing Question_data_1/MATHS -Advance.docx. Results appended to outputs_70b/MATHS -Advance_results.txt

Processing Maths-BASIC statements.docx with 108 sentences.


...

Finished processing Question_data_1/Physics- BASIC.docx. Results appended to outputs_70b/Physics- BASIC_results.txt

Processing Physics-Intermediate.docx with 100 sentences.



Processing Files:  83%|████████▎ | 5/6 [2:02:05<24:20, 1460.22s/it]  

Finished processing Question_data_1/Physics-Intermediate.docx. Results appended to outputs_70b/Physics-Intermediate_results.txt

Processing Physics-adv.docx with 101 sentences.



Processing Files: 100%|██████████| 6/6 [2:26:56<00:00, 1469.49s/it]  

Finished processing Question_data_1/Physics-adv.docx. Results appended to outputs_70b/Physics-adv_results.txt

